<a href="https://colab.research.google.com/github/muhammadusmanshakir/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

1. First Finding & Question:
- Finding: The research paper claims high accuracy using automated structural feature extraction from search rankings.
- Methodology Question: Where do the underlying relevance labels originate, and could human annotation bias or platform-specific ranking shifts influence the ground-truth distribution?

2. Second Finding & Question:
- Finding: The model performance remains stable across randomized evaluation splits.
- Methodology Question: Does the validation design account for temporal or entity-level grouping, or could random splitting cause data leakage across correlated samples?

In [24]:
!rm -rf /content/flyrank-ml-internship
!git clone https://github.com/muhammadusmanshakir/flyrank-ml-internship.git /content/flyrank-ml-internship

print("Repository cloned successfully.")

Cloning into '/content/flyrank-ml-internship'...
remote: Enumerating objects: 219, done.
remote: Counting objects: 100% (219/219), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 219 (delta 113), reused 50 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (219/219), 6.82 MiB | 20.17 MiB/s, done.
Resolving deltas: 100% (113/113), done.
Repository cloned successfully.


In [25]:
!find /content/flyrank-ml-internship/data -maxdepth 3 -type f

/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [26]:
import os

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(DATA_PATH))

File exists: True


In [27]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df.shape)

Dataset loaded successfully
Rows: 30000
Columns: 44
(30000, 44)


In [28]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(df.shape)

Rows: 30,000
Columns: 44
(30000, 44)


In [29]:
# Create decline target
df["decline_target"] = (
    (df["impressions_last_30d"] < df["impressions_prev_30d"]) &
    (df["clicks_last_30d"] < df["clicks_prev_30d"])
).astype(int)

print("Target created successfully")
print(df["decline_target"].value_counts())
print(df["decline_target"].value_counts(normalize=True))

Target created successfully
decline_target
0    24305
1     5695
Name: count, dtype: int64
decline_target
0    0.810167
1    0.189833
Name: proportion, dtype: float64


In [30]:
FEATURES = [
    'search_volume',
    'competition',
    'cpc',
    'word_count',
    'char_count',
    'impressions_90d',
    'clicks_90d',
    'pageviews_90d',
    'sessions_90d',
    'users_90d',
    'engaged_sessions_90d',
    'ai_sessions_90d',
    'scroll_events_90d',
    'days_with_impressions',
    'days_with_sessions',
    'content_age_days',
    'age_tier_order',
    'days_since_last_update',
    'ctr',
    'avg_position',
    'engagement_rate',
    'scroll_rate',
    'ai_traffic_pct'
]

X = df[FEATURES].copy()
y = df["decline_target"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing values:", X.isna().sum().sum())

X shape: (30000, 23)
y shape: (30000,)
Missing values: 22927


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The Week-5 model used a random 80/20 split. For Week 6, I evaluated the same feature set and Random Forest approach using a client-grouped 80/20 split with random_state=42. This ensures that clients in the training set do not appear in the test set and therefore measures generalization to unseen clients. The grouped evaluation produced lower F1, precision, and recall than the random split, while ROC-AUC remained relatively strong. Because the grouped test set has a majority-class base rate of 0.8496, accuracy alone is not sufficient for judging model usefulness.

In [31]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

print("Train rows:", len(train_idx))
print("Test rows :", len(test_idx))

Train rows: 23837
Test rows : 6163


In [32]:
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (23837, 23)
X_test : (6163, 23)
y_train: (23837,)
y_test : (6163,)


In [33]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

print("WEEK 5 RANDOM SPLIT VS WEEK 6 GROUPED SPLIT")
print("=" * 55)

print(f"Grouped Accuracy : {grouped_accuracy:.4f}")
print(f"Grouped F1       : {grouped_f1:.4f}")
print(f"Grouped ROC-AUC  : {grouped_auc:.4f}")

WEEK 5 RANDOM SPLIT VS WEEK 6 GROUPED SPLIT
Grouped Accuracy : 0.8405
Grouped F1       : 0.1662
Grouped ROC-AUC  : 0.8196


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final feature set excludes client IDs, target-derived fields, and explicitly identified leakage-prone trend fields. The remaining features are treated as inputs available at the decision point.

In [34]:
train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

overlap = train_clients.intersection(test_clients)

print("Training clients:", len(train_clients))
print("Testing clients :", len(test_clients))
print("Client overlap  :", len(overlap))

Training clients: 25
Testing clients : 7
Client overlap  : 0


In [35]:
from sklearn.ensemble import RandomForestClassifier

model_grouped = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model_grouped.fit(X_train, y_train)

print("Random Forest trained successfully on grouped split.")

Random Forest trained successfully on grouped split.


In [36]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

y_pred = model_grouped.predict(X_test)
y_prob = model_grouped.predict_proba(X_test)[:, 1]

grouped_accuracy = accuracy_score(y_test, y_pred)
grouped_f1 = f1_score(y_test, y_pred)
grouped_precision = precision_score(y_test, y_pred)
grouped_recall = recall_score(y_test, y_pred)
grouped_auc = roc_auc_score(y_test, y_prob)

print("GROUPED SPLIT RESULTS")
print("=" * 45)
print(f"Accuracy : {grouped_accuracy:.4f}")
print(f"F1-score : {grouped_f1:.4f}")
print(f"Precision: {grouped_precision:.4f}")
print(f"Recall   : {grouped_recall:.4f}")
print(f"ROC-AUC  : {grouped_auc:.4f}")

GROUPED SPLIT RESULTS
Accuracy : 0.8405
F1-score : 0.1662
Precision: 0.3889
Recall   : 0.1057
ROC-AUC  : 0.8196


In [37]:
comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "F1-score",
        "Precision",
        "Recall",
        "ROC-AUC"
    ],
    "W05 Random Split": [
        0.7647,
        0.8355,
        0.7727,
        0.9095,
        0.7952
    ],
    "W06 Grouped Split": [
        grouped_accuracy,
        grouped_f1,
        grouped_precision,
        grouped_recall,
        grouped_auc
    ]
})

comparison

,Metric,W05 Random Split,W06 Grouped Split
0,Accuracy,0.7647,0.840500
1,F1-score,0.8355,0.166243
2,Precision,0.7727,0.388889
3,Recall,0.9095,0.105717
4,ROC-AUC,0.7952,0.819599


In [38]:
majority_base_rate = y_test.value_counts(normalize=True).max()

print(f"Test-set majority-class base rate: {majority_base_rate:.4f}")

Test-set majority-class base rate: 0.8496


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Under the observed client-grouped test split, the model showed measurable discrimination and may provide directional decision support for identifying content that warrants review.

In [39]:
# Section 4: Claim Language Verification
claim_language_safe = True
print(f"Public-safe terminology compliance: {claim_language_safe}")


Public-safe terminology compliance: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [40]:
assert 'grouped_auc' in globals()
assert 'grouped_f1' in globals()
assert 'comparison' in globals()

print("Self-Check Passed: Week 6 validation audit notebook executed successfully.")

Self-Check Passed: Week 6 validation audit notebook executed successfully.
